In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
DATA_DIR = Path("~/Thesis/project/data/processed").expanduser()

SPLITS_JSON   = DATA_DIR / "window/splits_freeze.json"
WIN_TRAIN_CSV = DATA_DIR / "window/windows_train.csv"
WIN_VAL_CSV   = DATA_DIR / "window/windows_val.csv"
WIN_TEST_CSV  = DATA_DIR / "window/windows_test.csv"

MEDIAPIPE_DIR = DATA_DIR / "mediapipe"

print("DATA_DIR:", DATA_DIR)
print("SPLITS_JSON:", SPLITS_JSON)
print("MEDIAPIPE_DIR:", MEDIAPIPE_DIR)

VIDEO_DIR = (Path("~/Thesis/project/data/raw/video").expanduser())
def video_path(video_id): 
    return VIDEO_DIR / f"{video_id}.mp4"

In [ ]:

for p in [SPLITS_JSON, WIN_TRAIN_CSV, WIN_VAL_CSV, WIN_TEST_CSV]:
    print(p, "->", "OK" if p.exists() else "MISSING")


In [ ]:
with open(SPLITS_JSON, "r") as f:
    splits = json.load(f)

LABELS = ["MF", "SK", "SJ"]
WINDOW_SEC = float(splits["window_sec"])
HOP_SEC    = float(splits["hop_sec"])

print("Train/Val/Test videos:",
      len(splits["train_videos"]),
      len(splits["val_videos"]),
      len(splits["test_videos"]))
print("Window_sec / Hop_sec:", WINDOW_SEC, HOP_SEC)


In [ ]:
def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def standardize_windows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    vid_col = _find_col(df, ["video_id", "video", "vid", "Video", "mp_id"])
    if vid_col is None:
        raise ValueError(f"Could not find video id column. Columns: {list(df.columns)[:30]}")
    df.rename(columns={vid_col: "video_id"}, inplace=True)

    start_col = _find_col(df, ["w_start", "start", "start_sec", "t0", "window_start", "win_start"])
    end_col   = _find_col(df, ["w_end", "end", "end_sec", "t1", "window_end", "win_end"])
    if start_col is None or end_col is None:
        raise ValueError(f"Could not find start/end columns. Columns: {list(df.columns)[:30]}")
    df.rename(columns={start_col: "w_start", end_col: "w_end"}, inplace=True)

    mask_col = _find_col(df, ["mask", "valid", "is_valid", "keep", "train_mask"])
    if mask_col is None:
        df["mask"] = 1
    else:
        df.rename(columns={mask_col: "mask"}, inplace=True)

    for y in LABELS:
        if y not in df.columns:
            alt = _find_col(df, [y.lower(), f"y_{y}", f"label_{y}"])
            if alt is None:
                raise ValueError(f"Missing label column {y}.")
            df.rename(columns={alt: y}, inplace=True)

    df["w_start"] = pd.to_numeric(df["w_start"], errors="coerce")
    df["w_end"]   = pd.to_numeric(df["w_end"], errors="coerce")
    df["mask"]    = pd.to_numeric(df["mask"], errors="coerce").fillna(0).astype(int)
    for y in LABELS:
        df[y] = pd.to_numeric(df[y], errors="coerce").fillna(0).astype(int)

    df = df[(df["w_end"] > df["w_start"]) & df["w_start"].notna() & df["w_end"].notna()]
    return df.reset_index(drop=True)

win_train = standardize_windows(pd.read_csv(WIN_TRAIN_CSV))
win_val   = standardize_windows(pd.read_csv(WIN_VAL_CSV))
win_test  = standardize_windows(pd.read_csv(WIN_TEST_CSV))

print("Window rows:", len(win_train), len(win_val), len(win_test))
win_train.head()


In [ ]:
ALL_VIDS = splits["train_videos"] + splits["val_videos"] + splits["test_videos"]
missing = [vid for vid in ALL_VIDS if not (MEDIAPIPE_DIR / f"{vid}.csv").exists()]
print("Missing MediaPipe CSVs:", len(missing))
if missing:
    print(missing[:30])


In [ ]:
def standardize_mediapipe(df: pd.DataFrame):
    df = df.copy()

    ts_col = _find_col(df, ["timestamp_sec", "time_sec", "t_sec", "timestamp", "time"])
    if ts_col is None:
        raise ValueError("No timestamp column found")
    df.rename(columns={ts_col: "timestamp_sec"}, inplace=True)

    fi_col = _find_col(df, ["frame_idx", "frame", "idx"])
    if fi_col and fi_col != "frame_idx":
        df.rename(columns={fi_col: "frame_idx"}, inplace=True)

    if "video_path" in df.columns:
        df.drop(columns=["video_path"], inplace=True)

    drop_cols = []
    if "tongueOut" in df.columns:
        drop_cols.append("tongueOut")
    drop_cols += [c for c in df.columns if c.startswith("cheekSquint")]

    all_null = [c for c in df.columns
                if c not in ["timestamp_sec", "frame_idx"] and df[c].isna().all()]
    drop_cols = sorted(set(drop_cols + all_null))
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

    df["timestamp_sec"] = pd.to_numeric(df["timestamp_sec"], errors="coerce")
    df = df[df["timestamp_sec"].notna()].sort_values("timestamp_sec").reset_index(drop=True)

    keep = ["timestamp_sec"]
    if "frame_idx" in df.columns:
        keep.append("frame_idx")

    feat_cols = [c for c in df.columns if c not in keep and pd.api.types.is_numeric_dtype(df[c])]
    df = df[keep + feat_cols].copy()

    df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df, feat_cols


def load_mediapipe_csv(video_id: str, mediapipe_dir: Path):
    path = mediapipe_dir / f"{video_id}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing mediapipe file: {path}")
    df = pd.read_csv(path)
    df, feat_cols = standardize_mediapipe(df)
    return df, feat_cols


In [ ]:
def build_global_feature_set(video_ids, mediapipe_dir: Path):
    common = None
    for vid in video_ids:
        df, feats = load_mediapipe_csv(vid, mediapipe_dir)
        s = set(feats)
        common = s if common is None else (common & s)
    common = sorted(common) if common is not None else []
    return common

GLOBAL_FEATS = build_global_feature_set(ALL_VIDS, MEDIAPIPE_DIR)

In [ ]:
def estimate_fps(timestamp_sec: np.ndarray) -> float:
    t = np.asarray(timestamp_sec, dtype=np.float32)
    dt = np.diff(t)
    dt = dt[(dt > 1e-6) & np.isfinite(dt)]
    if len(dt) == 0:
        return 10.0
    return float(1.0 / np.median(dt))

def extract_window_sequence(
    mp_df: pd.DataFrame,
    feat_cols: list[str],
    w_start: float,
    w_end: float,
    T: int,
    fill_value: float = 0.0
) -> np.ndarray:

    sub = mp_df[(mp_df["timestamp_sec"] >= w_start) & (mp_df["timestamp_sec"] < w_end)]
    if len(sub) < 2:
        return np.full((T, len(feat_cols)), fill_value, dtype=np.float32)

    t = sub["timestamp_sec"].values.astype(np.float32)
    X = sub[feat_cols].values.astype(np.float32)  # (n, D)

    tt = np.linspace(float(w_start), float(w_end), num=T, endpoint=False, dtype=np.float32)

    out = np.empty((T, X.shape[1]), dtype=np.float32)
    for j in range(X.shape[1]):
        out[:, j] = np.interp(tt, t, X[:, j]).astype(np.float32)

    return out


In [ ]:
def fit_video_zscore(mp_df: pd.DataFrame, feat_cols: list[str]):
    X = mp_df[feat_cols].values.astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return mu, sd

def apply_video_zscore(seq: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    return (seq - mu[None, :]) / sd[None, :]


In [ ]:
class MediaPipeCache:
    def __init__(self, mediapipe_dir: Path, global_feats: list[str]):
        self.dir = mediapipe_dir
        self.global_feats = global_feats
        self._store = {}  # vid -> dict(df, mu, sd, fps_est)

    def get(self, video_id: str):
        if video_id in self._store:
            return self._store[video_id]

        df, feats = load_mediapipe_csv(video_id, self.dir)

        missing = [c for c in self.global_feats if c not in df.columns]
        if missing:
            raise ValueError(f"{video_id}: missing global features: {missing[:10]} ...")

        fps = estimate_fps(df["timestamp_sec"].values)
        mu, sd = fit_video_zscore(df, self.global_feats)

        pack = {"df": df, "mu": mu, "sd": sd, "fps": fps}
        self._store[video_id] = pack
        return pack

cache = MediaPipeCache(MEDIAPIPE_DIR, GLOBAL_FEATS)


In [ ]:
PLOT_FACE = [
    "browDownLeft", "browInnerUp",
    "eyeBlinkLeft", "eyeSquintLeft",
    "mouthSmileLeft", "mouthPressLeft",
    "jawOpen",
]
PLOT_BODY = [
    "leftShoulder_y", "rightShoulder_y",
    "leftWrist_y", "rightWrist_y",
    "leftElbow_y", "rightElbow_y",
]

PLOT_FACE = [c for c in PLOT_FACE if c in GLOBAL_FEATS]
PLOT_BODY = [c for c in PLOT_BODY if c in GLOBAL_FEATS]
PLOT_FEATS = PLOT_FACE + PLOT_BODY

plot_idx = [GLOBAL_FEATS.index(c) for c in PLOT_FEATS]
print("Plot feats:", PLOT_FEATS)

T_PLOT = 40
t_plot = np.linspace(0, WINDOW_SEC, num=T_PLOT, endpoint=False)

def aggregate_over_split(
    windows_df: pd.DataFrame,
    video_ids: list[str],
    label: str,
    max_windows_per_video: int = 1500
):
    seqs = []
    total = 0

    for vid in video_ids:
        wv = windows_df[(windows_df["video_id"] == vid) & (windows_df["mask"] == 1) & (windows_df[label] == 1)]
        if len(wv) == 0:
            continue
        if len(wv) > max_windows_per_video:
            wv = wv.sample(max_windows_per_video, random_state=0)

        pack = cache.get(vid)
        mp_df = pack["df"]
        mu, sd = pack["mu"], pack["sd"]

        for _, row in wv.iterrows():
            seq = extract_window_sequence(mp_df, GLOBAL_FEATS, float(row["w_start"]), float(row["w_end"]), T=T_PLOT)
            seq = apply_video_zscore(seq, mu, sd)
            seqs.append(seq)

        total += len(wv)

    if total == 0:
        return None, None, 0

    X = np.stack(seqs, axis=0)
    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    return mean, std, total

TEST_VIDS = splits["test_videos"]

agg = {}
counts = {}
for y in LABELS:
    mean, std, n = aggregate_over_split(win_test, TEST_VIDS, label=y)
    agg[y] = (mean, std)
    counts[y] = n

print("Counts:", counts)

def plot_aggregate_by_label():
    for y in LABELS:
        mean, std = agg[y]
        if mean is None:
            print(f"Skip {y}: no windows")
            continue

        m = mean[:, plot_idx]
        s = std[:, plot_idx]

        plt.figure(figsize=(12, 6))
        for j, name in enumerate(PLOT_FEATS):
            plt.plot(t_plot, m[:, j], label=name)
            plt.fill_between(t_plot, m[:, j] - s[:, j], m[:, j] + s[:, j], alpha=0.15)

        plt.title(f"TEST aggregate trajectories for {y} (N={counts[y]})")
        plt.xlabel("Time within 4s window (sec)")
        plt.ylabel("Z-scored activation (per video)")
        plt.legend(ncol=2, fontsize=9)
        plt.tight_layout()
        plt.show()

plot_aggregate_by_label()


In [ ]:
T_MODEL = 40

class VideoWindowsDatasetFixedT(Dataset):
    def __init__(self, windows_df: pd.DataFrame, cache: MediaPipeCache, T_model: int):
        self.w = windows_df.copy().reset_index(drop=True)
        self.cache = cache
        self.T_model = T_model

    def __len__(self):
        return len(self.w)

    def __getitem__(self, idx):
        row = self.w.iloc[idx]
        vid = row["video_id"]

        pack = self.cache.get(vid)
        mp_df = pack["df"]
        mu, sd = pack["mu"], pack["sd"]

        seq = extract_window_sequence(
            mp_df, self.cache.global_feats,
            float(row["w_start"]), float(row["w_end"]),
            T=self.T_model
        )
        seq = apply_video_zscore(seq, mu, sd)

        x = torch.from_numpy(seq)  # (T, D)
        y = torch.tensor([int(row[l]) for l in LABELS], dtype=torch.float32)
        m = torch.tensor(int(row["mask"]), dtype=torch.float32)

        return x, y, m

class GRUWindowClassifier(nn.Module):
    def __init__(self, in_dim, hidden=128, num_layers=1, dropout=0.1, out_dim=3, head_dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=in_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(head_dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        out, _ = self.gru(x)
        pooled = out.mean(dim=1)
        return self.head(pooled)


def compute_pos_weight(windows_df: pd.DataFrame):
    df = windows_df[windows_df["mask"] == 1]
    pos = df[LABELS].sum().values.astype(np.float32)
    neg = (len(df) - pos).astype(np.float32)
    pos = np.maximum(pos, 1.0)
    return torch.tensor(neg / pos, dtype=torch.float32)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, return_preds: bool = False):
    model.eval()
    all_y, all_p, all_m, all_logits = [], [], [], []
    total_loss = 0.0
    total_n = 0.0

    for x, y, m in loader:
        x = x.to(device)
        y = y.to(device)
        m = m.to(device)

        logits = model(x)  # (B,3)
        loss_mat = criterion(logits, y)          # (B,3)
        loss = (loss_mat * m[:, None]).sum() / (m.sum() + 1e-6)

        total_loss += float(loss.item()) * float(m.sum().item())
        total_n += float(m.sum().item())

        probs = torch.sigmoid(logits)

        all_logits.append(logits.detach().cpu().numpy())
        all_p.append(probs.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())
        all_m.append(m.detach().cpu().numpy())

    Y = np.concatenate(all_y, axis=0)
    P = np.concatenate(all_p, axis=0)
    M = np.concatenate(all_m, axis=0)
    L = np.concatenate(all_logits, axis=0)

    keep = (M > 0.5)
    Yk, Pk = Y[keep], P[keep]

    ap, auc = {}, {}
    for i, lab in enumerate(LABELS):
        y_true = Yk[:, i]
        y_score = Pk[:, i]

        # safety: filter non-finite
        finite = np.isfinite(y_score)
        y_true = y_true[finite]
        y_score = y_score[finite]

        if len(y_true) == 0 or len(np.unique(y_true)) < 2:
            ap[lab] = np.nan
            auc[lab] = np.nan
        else:
            ap[lab] = float(average_precision_score(y_true, y_score))
            auc[lab] = float(roc_auc_score(y_true, y_score))

    loss_out = (total_loss / (total_n + 1e-6))

    if return_preds:
        return loss_out, ap, auc, L, P, Y, M
    return loss_out, ap, auc


@torch.no_grad()
def predict_logits_probs(model, loader, device):
    model.eval()
    all_logits, all_probs, all_y, all_m = [], [], [], []

    for x, y, m in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)

        all_logits.append(logits.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())
        all_m.append(m.detach().cpu().numpy())

    L = np.concatenate(all_logits, axis=0).astype(np.float32)
    P = np.concatenate(all_probs, axis=0).astype(np.float32)
    Y = np.concatenate(all_y, axis=0).astype(np.float32)
    M = np.concatenate(all_m, axis=0).astype(np.float32)
    return L, P, Y, M


def export_unimodal_preds(
    out_root: Path,
    split: str,
    modality: str,
    logits: np.ndarray,
    probs: np.ndarray,
    mask: np.ndarray,
):
  
    out = Path(out_root) / split / modality
    out.mkdir(parents=True, exist_ok=True)

    np.save(out / f"logits_{split}.npy", logits.astype(np.float32))
    np.save(out / f"probs_{split}.npy", probs.astype(np.float32))

    conf = np.nanmax(probs, axis=1).astype(np.float32)
    present = (mask > 0.5).astype(np.float32)

    np.save(out / f"conf_{split}.npy", conf)
    np.save(out / f"present_{split}.npy", present)

    print(f"Saved -> {out}")

def train_epoch(model, loader, criterion, opt, device):
    model.train()
    total_loss = 0.0
    total_n = 0.0

    for x, y, m in loader:
        x = x.to(device)
        y = y.to(device)
        m = m.to(device)

        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss_mat = criterion(logits, y)
        loss = (loss_mat * m[:, None]).sum() / (m.sum() + 1e-6)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total_loss += float(loss.item()) * float(m.sum().item())
        total_n += float(m.sum().item())

    return total_loss / (total_n + 1e-6)


In [ ]:
ds_train = VideoWindowsDatasetFixedT(win_train, cache, T_MODEL)
ds_val   = VideoWindowsDatasetFixedT(win_val,   cache, T_MODEL)
ds_test  = VideoWindowsDatasetFixedT(win_test,  cache, T_MODEL)

BATCH = 64

train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=0)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = GRUWindowClassifier(
    in_dim=len(GLOBAL_FEATS),
    hidden=128,
    num_layers=1,
    out_dim=len(LABELS),
).to(device)

pos_weight = compute_pos_weight(win_train)
pos_weight = torch.clamp(pos_weight, max=10.0).to(device)

criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)


opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=2e-2)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode="max",
    factor=0.5,
    patience=3,
    threshold=1e-3,
    cooldown=1,
    min_lr=1e-6,
)

print("Device:", device)
print("pos_weight:", dict(zip(LABELS, pos_weight.detach().cpu().numpy().round(3))))

def mean_ap(ap_dict, labels=None):
    """Macro mean AP across labels (ignores NaNs)."""
    if labels is None:
        labels = list(ap_dict.keys())
    vals = [ap_dict[l] for l in labels if (l in ap_dict and np.isfinite(ap_dict[l]))]
    return float(np.mean(vals)) if len(vals) else float("nan")


EPOCHS = 30
PATIENCE = 8
WARMUP_EPOCHS = 6
MIN_DELTA = 0.002

best = {"score": -1.0, "epoch": -1, "state": None}
bad = 0

for ep in range(1, EPOCHS + 1):
    tr_loss = train_epoch(model, train_loader, criterion, opt, device)
    va_loss, va_ap, va_auc = eval_epoch(model, val_loader, criterion, device)

    score = mean_ap(va_ap, labels=LABELS)

    scheduler.step(score)

    lr_now = opt.param_groups[0]["lr"]
    print(f"Epoch {ep:02d} | lr {lr_now:.2e} | train loss {tr_loss:.4f} | val loss {va_loss:.4f} | meanAP {score:.4f}")
    print("  Val AP :", va_ap)
    print("  Val AUC:", va_auc)

    improved = (score > best["score"] + MIN_DELTA)

    if improved:
        best["score"] = score
        best["epoch"] = ep
        best["state"] = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        bad = 0
    else:
        if ep > WARMUP_EPOCHS:
            bad += 1
            if bad >= PATIENCE:
                print(f"Early stopping. Best epoch={best['epoch']} meanAP={best['score']:.4f}")
                break

assert best["state"] is not None, "No best state saved — something went wrong."
model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})


te_loss, te_ap, te_auc = eval_epoch(model, test_loader, criterion, device)
print("\nBEST TEST")
print("Best epoch:", best["epoch"], "best meanAP:", best["score"])
print("TEST loss:", te_loss)
print("TEST AP :", te_ap)
print("TEST AUC:", te_auc)



In [ ]:
OUT_ROOT = Path("preds_unimodal")
MODALITY = "video"

export_train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=False, num_workers=0)
export_val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=0)
export_test_loader  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=0)


model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})
model.eval()

Ltr, Ptr, Ytr, Mtr = predict_logits_probs(model, export_train_loader, device)
Lva, Pva, Yva, Mva = predict_logits_probs(model, export_val_loader, device)
Lte, Pte, Yte, Mte = predict_logits_probs(model, export_test_loader, device)

export_unimodal_preds(OUT_ROOT, "train", MODALITY, Ltr, Ptr, Mtr)
export_unimodal_preds(OUT_ROOT, "val",   MODALITY, Lva, Pva, Mva)
export_unimodal_preds(OUT_ROOT, "test",  MODALITY, Lte, Pte, Mte)

print("Shapes:")
print(" train logits", Ltr.shape, "probs", Ptr.shape, "mask", Mtr.shape)
print(" val   logits", Lva.shape, "probs", Pva.shape, "mask", Mva.shape)
print(" test  logits", Lte.shape, "probs", Pte.shape, "mask", Mte.shape)


In [ ]:
OUT_DIR = DATA_DIR / "models" / "video_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = OUT_DIR / "gru_video_only.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "global_feats": GLOBAL_FEATS,
        "T_model": T_MODEL,
        "splits": splits,
        "pos_weight": pos_weight.detach().cpu().numpy(),
    },
    ckpt_path
)

print("Saved:", ckpt_path)


In [ ]:
def _set_dropout_p(module, p):
    drops = []
    for m in module.modules():
        if isinstance(m, torch.nn.Dropout):
            drops.append((m, m.p))
            m.p = p
    return drops

def _restore_dropout(drops):
    for m, oldp in drops:
        m.p = oldp

def compute_global_gradient_saliency(model, loader, device, labels=LABELS, use_grad_x_input=True, max_batches=None):
    was_training = model.training
    model.train()
    drops = _set_dropout_p(model, 0.0)

    sal_sum = {lab: None for lab in labels}
    n_batches = 0
    T = D = None

    for b, (x, y, m) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break

        keep = (m > 0.5)
        if keep.sum().item() == 0:
            continue

        x = x[keep].to(device)

        if T is None:
            T, D = x.shape[1], x.shape[2]
            for lab in labels:
                sal_sum[lab] = torch.zeros((T, D), device=device)

        x = x.detach().requires_grad_(True)
        probs = torch.sigmoid(model(x))

        for i, lab in enumerate(labels):
            model.zero_grad(set_to_none=True)
            if x.grad is not None:
                x.grad.zero_()

            probs[:, i].sum().backward(retain_graph=True)

            if use_grad_x_input:
                sal = (x.grad.abs() * x.abs())
            else:
                sal = x.grad.abs()

            sal_sum[lab] += sal.mean(dim=0)

        n_batches += 1

    _restore_dropout(drops)
    if not was_training:
        model.eval()

    sal_TD, feat_imp, time_imp = {}, {}, {}
    for lab in labels:
        S = (sal_sum[lab] / max(n_batches, 1)).detach().cpu().numpy()
        sal_TD[lab] = S
        feat_imp[lab] = S.mean(axis=0)
        time_imp[lab] = S.mean(axis=1)
    return sal_TD, feat_imp, time_imp

test_loader_sal = DataLoader(ds_test, batch_size=64, shuffle=False, num_workers=0)
sal_TD_test, feat_imp_test, time_imp_test = compute_global_gradient_saliency(
    model, test_loader_sal, device=device, labels=LABELS, use_grad_x_input=True
)

def plot_top_features(feat_imp, lab, K=20):
    imp = feat_imp[lab]
    idx = np.argsort(-imp)[:K]
    names = [GLOBAL_FEATS[i] for i in idx]
    vals = imp[idx]
    plt.figure(figsize=(10,5))
    plt.bar(range(K), vals)
    plt.xticks(range(K), names, rotation=60, ha="right")
    plt.title(f"Global gradient saliency (TEST): Top-{K} features for {lab}")
    plt.ylabel("mean |grad*x|")
    plt.tight_layout()
    plt.show()

for lab in LABELS:
    plot_top_features(feat_imp_test, lab, K=20)

t_axis = np.linspace(0, WINDOW_SEC, num=T_MODEL, endpoint=False)
plt.figure(figsize=(10,4))
for lab in LABELS:
    plt.plot(t_axis, time_imp_test[lab], label=lab)
plt.title("Global gradient saliency over time within 4s window (TEST)")
plt.xlabel("time within window (sec)")
plt.ylabel("mean |grad*x|")
plt.legend()
plt.tight_layout()
plt.show()

@torch.no_grad()
def collect_XYM(loader):
    Xs, Ys, Ms = [], [], []
    for x, y, m in loader:
        Xs.append(x.numpy())
        Ys.append(y.numpy())
        Ms.append(m.numpy())
    X = np.concatenate(Xs, axis=0)
    Y = np.concatenate(Ys, axis=0)
    M = np.concatenate(Ms, axis=0)
    keep = (M > 0.5)
    return X[keep], Y[keep]

@torch.no_grad()
def predict_probs_np(model, X_np, device):
    X = torch.from_numpy(X_np).float().to(device)
    return torch.sigmoid(model(X)).cpu().numpy()

def permutation_importance_test(model, loader, device, feat_imp_test, K=150, seed=0):
    rng = np.random.default_rng(seed)
    X, Y = collect_XYM(loader)
    P0 = predict_probs_np(model, X, device)

    base_ap = {}
    for i, lab in enumerate(LABELS):
        base_ap[lab] = float(average_precision_score(Y[:, i], P0[:, i])) if len(np.unique(Y[:, i])) > 1 else np.nan

    s = np.zeros(len(GLOBAL_FEATS), dtype=np.float32)
    for lab in LABELS:
        s += feat_imp_test[lab].astype(np.float32)
    cand = np.argsort(-s)[:K]

    out = {lab: [] for lab in LABELS}
    for j in cand:
        Xp = X.copy()
        perm = rng.permutation(Xp.shape[0])
        Xp[:, :, j] = Xp[perm, :, j]
        Pp = predict_probs_np(model, Xp, device)
        for i, lab in enumerate(LABELS):
            if not np.isfinite(base_ap[lab]) or len(np.unique(Y[:, i])) < 2:
                continue
            ap = float(average_precision_score(Y[:, i], Pp[:, i]))
            out[lab].append((GLOBAL_FEATS[j], base_ap[lab] - ap))

    for lab in LABELS:
        out[lab] = sorted(out[lab], key=lambda x: -x[1])
    return base_ap, out

test_loader_pi = DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=0)
base_ap, pi = permutation_importance_test(model, test_loader_pi, device, feat_imp_test, K=150, seed=0)

print("Base TEST AP:", base_ap)
for lab in LABELS:
    top = pi[lab][:20]
    plt.figure(figsize=(10,5))
    plt.bar(range(len(top)), [d for _, d in top])
    plt.xticks(range(len(top)), [n for n,_ in top], rotation=60, ha="right")
    plt.title(f"Permutation importance (TEST): Top-20 AP drop for {lab}")
    plt.ylabel("AP drop when permuted")
    plt.tight_layout()
    plt.show()
